In [2]:
from pathlib import Path
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").exists() else NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_dataset_root

In [3]:
DATASET_ROOT = get_dataset_root()

OUTPUT_ROOT = Path("../outputs/roi_dataset_v3")

GENUINE_OUTPUT = OUTPUT_ROOT / "genuine"
FORGED_OUTPUT = OUTPUT_ROOT / "forged"

GENUINE_OUTPUT.mkdir(parents=True, exist_ok=True)
FORGED_OUTPUT.mkdir(parents=True, exist_ok=True)

ROI_SIZE = 224


In [4]:
def show_image(title, image, cmap=None, figsize=(8, 10)):
    plt.figure(figsize=figsize)
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()

In [5]:
def contour_stamp_score(contour):
    area = cv2.contourArea(contour)

    if area <= 0:
        return None

    x, y, w, h = cv2.boundingRect(contour)
    bbox_area = w * h

    if bbox_area <= 0:
        return None

    aspect_ratio = w / h
    compactness = min(w, h) / max(w, h)
    extent = area / bbox_area

    perimeter = cv2.arcLength(contour, True)

    if perimeter <= 0:
        circularity = 0
    else:
        circularity = (4 * np.pi * area) / (perimeter ** 2)
    if area < 200:
        return None

    if compactness < 0.35:
        return None
    score = (
        0.45 * compactness +
        0.35 * circularity +
        0.20 * extent
    )

    return {
        "contour": contour,
        "x": x,
        "y": y,
        "w": w,
        "h": h,
        "area": area,
        "aspect_ratio": aspect_ratio,
        "compactness": compactness,
        "extent": extent,
        "circularity": circularity,
        "score": score
    }

In [6]:
def find_stamp_candidates_by_contours(image_bgr, display_width=900):
    h0, w0 = image_bgr.shape[:2]

    scale = display_width / w0
    new_h = int(h0 * scale)

    image_resized = cv2.resize(image_bgr, (display_width, new_h))

    image_hsv = cv2.cvtColor(image_resized, cv2.COLOR_BGR2HSV)

    h = image_hsv[:, :, 0]
    s = image_hsv[:, :, 1]
    v = image_hsv[:, :, 2]

    hue_mask = cv2.inRange(h, 90, 170)
    sat_mask = cv2.inRange(s, 25, 255)
    val_mask = cv2.inRange(v, 30, 255)

    mask = cv2.bitwise_and(hue_mask, sat_mask)
    mask = cv2.bitwise_and(mask, val_mask)

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))

    mask_opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)
    mask_cleaned = cv2.morphologyEx(mask_opened, cv2.MORPH_CLOSE, kernel_close)

    contours, _ = cv2.findContours(
        mask_cleaned,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    candidates = []

    for contour in contours:
        result = contour_stamp_score(contour)

        if result is not None:
            candidates.append(result)

    candidates = sorted(
        candidates,
        key=lambda item: item["score"],
        reverse=True
    )

    return image_resized, mask_cleaned, candidates

In [7]:
def extract_roi_from_candidate_original(
    original_image,
    resized_image,
    candidate,
    pad_factor=1.0
):
    original_h, original_w = original_image.shape[:2]
    resized_h, resized_w = resized_image.shape[:2]

    scale_x = original_w / resized_w
    scale_y = original_h / resized_h

    x, y, w, h = candidate["x"], candidate["y"], candidate["w"], candidate["h"]

    pad_x = int(w * pad_factor)
    pad_y = int(h * pad_factor)
    x1 = max(x - pad_x, 0)
    y1 = max(y - pad_y, 0)
    x2 = min(x + w + pad_x, resized_w)
    y2 = min(y + h + pad_y, resized_h)
    ox1 = int(x1 * scale_x)
    oy1 = int(y1 * scale_y)
    ox2 = int(x2 * scale_x)
    oy2 = int(y2 * scale_y)

    roi_original = original_image[oy1:oy2, ox1:ox2]

    return roi_original, (ox1, oy1, ox2, oy2)

In [8]:
def save_roi_image(roi_bgr, output_path):

    success = cv2.imwrite(
        str(output_path),
        roi_bgr
    )

    return success

In [9]:
all_images = list(DATASET_ROOT.rglob("*.png"))

print("Total images:", len(all_images))

Total images: 531


In [10]:
all_images = list(DATASET_ROOT.rglob("*.png"))
print("Total images:", len(all_images))

records = []
expected_columns = [
    "source_path",
    "roi_path",
    "class_name",
    "roi_saved",
    "reason",
    "candidate_score",
    "candidate_x",
    "candidate_y",
    "candidate_w",
    "candidate_h",
    "bbox",
    "num_candidates",
    "selected_candidate_rank",
]

for image_path in tqdm(all_images, desc="Extracting ROIs"):
    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        records.append({
            "source_path": str(image_path),
            "roi_saved": False,
            "reason": "image_read_failed"
        })
        continue

    image_resized, mask_cleaned, candidates = find_stamp_candidates_by_contours(image_bgr)

    if len(candidates) == 0:
        records.append({
            "source_path": str(image_path),
            "roi_saved": False,
            "reason": "no_candidate_found"
        })
        continue

    best_candidate = candidates[0]

    if best_candidate is None:
        records.append({
            "source_path": str(image_path),
            "roi_saved": False,
            "reason": "no_candidate_selected"
        })
        continue

    roi, bbox = extract_roi_from_candidate_original(
        original_image=image_bgr,
        resized_image=image_resized,
        candidate=best_candidate,
        pad_factor=1.0
    )

    if roi is None or roi.size == 0:
        records.append({
            "source_path": str(image_path),
            "roi_saved": False,
            "reason": "empty_roi"
        })
        continue

    roi_standardized = roi

    class_name = "genuine" if "class_0_genuine" in str(image_path) else "forged"
    output_dir = GENUINE_OUTPUT if class_name == "genuine" else FORGED_OUTPUT

    output_name = image_path.stem + "_roi.png"
    output_path = output_dir / output_name

    success = save_roi_image(roi_standardized, output_path)

    records.append({
        "source_path": str(image_path),
        "roi_path": str(output_path),
        "class_name": class_name,
        "roi_saved": success,
        "reason": "success" if success else "save_failed",
        "candidate_score": best_candidate["score"],
        "candidate_x": best_candidate["x"],
        "candidate_y": best_candidate["y"],
        "candidate_w": best_candidate["w"],
        "candidate_h": best_candidate["h"],
        "bbox": bbox,
        "num_candidates": len(candidates),
        "selected_candidate_rank": 0,
    })

df_roi = pd.DataFrame(records, columns=expected_columns)
if df_roi.empty:
    df_roi = pd.DataFrame(columns=expected_columns)

df_roi.head()

Total images: 531


Extracting ROIs: 100%|██████████| 531/531 [02:39<00:00,  3.34it/s]


,source_path,roi_path,class_name,roi_saved,reason,candidate_score,candidate_x,candidate_y,candidate_w,candidate_h,bbox,num_candidates,selected_candidate_rank
0,D:\sem-07\EC-7205-computer-vision-image-proces...,..\outputs\roi_dataset_v3\forged\Forged_Laser_...,forged,True,success,0.764262,644.0,1083.0,70.0,81.0,"(1582, 2763, 2161, 3434)",1.0,0.0
1,D:\sem-07\EC-7205-computer-vision-image-proces...,..\outputs\roi_dataset_v3\forged\Forged_Laser_...,forged,True,success,0.841667,173.0,1004.0,80.0,83.0,"(256, 2540, 917, 3227)",1.0,0.0
2,D:\sem-07\EC-7205-computer-vision-image-proces...,..\outputs\roi_dataset_v3\forged\Forged_Laser_...,forged,True,success,0.836470,634.0,797.0,83.0,83.0,"(1518, 1969, 2205, 2656)",1.0,0.0
3,D:\sem-07\EC-7205-computer-vision-image-proces...,..\outputs\roi_dataset_v3\forged\Forged_Laser_...,forged,True,success,0.888471,676.0,722.0,84.0,85.0,"(1631, 1757, 2326, 2460)",1.0,0.0
4,D:\sem-07\EC-7205-computer-vision-image-proces...,..\outputs\roi_dataset_v3\forged\Forged_Laser_...,forged,True,success,0.505128,645.0,1086.0,69.0,78.0,"(3175, 5557, 4316, 6847)",1.0,0.0


In [11]:
print("ROI extraction summary:")
if "roi_saved" in df_roi.columns:
    print(df_roi["roi_saved"].value_counts())
else:
    print("No ROI records available. Re-run the extraction cell.")

print("\nClass-wise summary:")
if {"class_name", "roi_saved"}.issubset(df_roi.columns):
    print(df_roi.groupby(["class_name", "roi_saved"]).size())
else:
    print("No class-wise summary available yet.")

ROI extraction summary:
roi_saved
True     507
False     24
Name: count, dtype: int64

Class-wise summary:
class_name  roi_saved
forged      True         197
genuine     True         310
dtype: int64
